# cyp51A SNP Analysis - v8.1 (SNPeff-annotated VCFs)

**Critical Fix**: Using SNPeff-annotated VCF files from collections #450 and #351 which contain proper EFF annotations for variant effect prediction.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
import re
import gxy

print("📊 cyp51A SNP Analysis v8.1 - Using SNPeff-annotated VCFs")
print("🔧 Fixed: Using collections #450 and #351 with EFF annotations")

## Step 1: Load GTF file and extract cyp51A gene information

In [ ]:
# Find GTF file
gtf_file = None
gtf_candidates = [4, 5, 6, 7, 8, 9, 10]  # Extended search

for hid in gtf_candidates:
    try:
        file_path = gxy.get(hid)
        if file_path and ('.gtf' in file_path.lower() or '.gff' in file_path.lower()):
            print(f"✓ Found GTF file at HID {hid}: {file_path}")
            gtf_file = file_path
            break
        else:
            print(f"⚠️ HID {hid} is not a GTF file: {file_path}")
    except Exception as e:
        print(f"⚠️ Could not access HID {hid}: {e}")

if not gtf_file:
    print("✗ No GTF file found")
    raise FileNotFoundError("GTF file not found")

print(f"📁 Using GTF file: {gtf_file}")

In [ ]:
# Parse GTF to find cyp51A gene (Afu4g06890)
cyp51a_info = None
cyp51a_exons = []

with open(gtf_file, 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        
        fields = line.strip().split('\t')
        if len(fields) < 9:
            continue
            
        chromosome, source, feature, start, end, score, strand, frame, attributes = fields
        
        # Look for cyp51A gene identifier
        if 'Afu4g06890' in attributes:
            if feature == 'gene':
                cyp51a_info = {
                    'chromosome': chromosome,
                    'start': int(start),
                    'end': int(end),
                    'strand': strand,
                    'attributes': attributes
                }
                print(f"✓ Found cyp51A gene: {chromosome}:{start}-{end} ({strand})")
            elif feature == 'exon' or feature == 'CDS':
                cyp51a_exons.append({
                    'feature': feature,
                    'start': int(start),
                    'end': int(end),
                    'attributes': attributes
                })

if not cyp51a_info:
    print("✗ cyp51A gene not found in GTF")
    raise ValueError("cyp51A gene not found")

print(f"📍 cyp51A location: {cyp51a_info['chromosome']}:{cyp51a_info['start']}-{cyp51a_info['end']}")
print(f"🧬 Found {len(cyp51a_exons)} exons/CDS features")

# Show exon structure
for i, exon in enumerate(cyp51a_exons[:5]):  # Show first 5
    print(f"  Exon {i+1}: {exon['start']}-{exon['end']} ({exon['feature']})")

## Step 2: Access SNPeff-annotated VCF files from collections #450 and #351

**Key Fix**: Using VCF files that have been processed by SNPeff and contain EFF annotations for variant effect prediction.

In [ ]:
# Get SNPeff-annotated VCF files from collections #450 and #351
# These are the correct VCFs with EFF annotations

snpeff_vcf_ids = [
    # From collection #450 (SNPeff-annotated)
    454, 455, 456, 457, 458, 459, 460, 461, 462,
    # From collection #351 (SNPeff-annotated) 
    355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368
]

print(f"🔬 Accessing {len(snpeff_vcf_ids)} SNPeff-annotated VCF files...")

# Verify these are actually VCF files with SNPeff annotations
valid_vcf_files = []
for hid in snpeff_vcf_ids[:3]:  # Test first 3
    try:
        file_path = gxy.get(hid)
        if file_path and '.vcf' in file_path.lower():
            # Quick check for EFF annotation in header
            with open(file_path, 'r') as f:
                for line_num, line in enumerate(f):
                    if line_num > 50:  # Only check first 50 lines of header
                        break
                    if 'ID=EFF' in line or 'SnpEff' in line:
                        print(f"✓ HID {hid} has SNPeff annotations")
                        valid_vcf_files.append((hid, file_path))
                        break
            if len(valid_vcf_files) == 0 or valid_vcf_files[-1][0] != hid:
                print(f"⚠️ HID {hid} may not have SNPeff annotations")
        else:
            print(f"⚠️ HID {hid} is not a VCF file: {file_path}")
    except Exception as e:
        print(f"⚠️ Could not access HID {hid}: {e}")

if len(valid_vcf_files) == 0:
    print("✗ No valid SNPeff-annotated VCF files found")
    # Fallback: try to access the collections directly
    print("🔄 Trying to access collections #450 and #351 directly...")
    for collection_id in [450, 351]:
        try:
            collection_path = gxy.get(collection_id)
            print(f"Collection {collection_id}: {collection_path}")
        except Exception as e:
            print(f"Could not access collection {collection_id}: {e}")
else:
    print(f"✓ Found {len(valid_vcf_files)} valid SNPeff-annotated VCF files")

## Step 3: Process SNPeff-annotated VCF files for cyp51A variants

Now using proper EFF annotations to identify functional variants.

In [ ]:
def parse_snpeff_effects(info_field):
    """Parse SNPeff EFF annotations from INFO field"""
    effects = []
    
    # Look for EFF= annotation
    eff_match = re.search(r'EFF=([^;]+)', info_field)
    if eff_match:
        eff_string = eff_match.group(1)
        # Parse individual effects (comma-separated)
        for effect in eff_string.split(','):
            # Format: EFFECT(IMPACT|FUNCTIONAL_CLASS|CODON_CHANGE|AMINO_ACID_CHANGE|...)
            effect_match = re.match(r'([^(]+)\(([^)]+)\)', effect)
            if effect_match:
                effect_type = effect_match.group(1)
                effect_details = effect_match.group(2).split('|')
                effects.append({
                    'effect': effect_type,
                    'impact': effect_details[0] if len(effect_details) > 0 else '',
                    'functional_class': effect_details[1] if len(effect_details) > 1 else '',
                    'codon_change': effect_details[2] if len(effect_details) > 2 else '',
                    'amino_acid_change': effect_details[3] if len(effect_details) > 3 else '',
                    'gene_name': effect_details[5] if len(effect_details) > 5 else ''
                })
    
    return effects

def is_cyp51a_variant(effects):
    """Check if variant affects cyp51A gene"""
    for effect in effects:
        if 'Afu4g06890' in effect.get('gene_name', '') or 'cyp51A' in effect.get('gene_name', '').lower():
            return True
    return False

def is_functional_variant(effects):
    """Check if variant has functional impact (non-synonymous)"""
    functional_effects = {
        'MISSENSE', 'NONSENSE', 'START_LOST', 'STOP_GAINED', 'STOP_LOST',
        'FRAME_SHIFT', 'CODON_INSERTION', 'CODON_DELETION', 'CODON_CHANGE_PLUS_CODON_INSERTION',
        'CODON_CHANGE_PLUS_CODON_DELETION', 'UTR_5_PRIME', 'UTR_3_PRIME', 'SPLICE_SITE_ACCEPTOR',
        'SPLICE_SITE_DONOR', 'START_GAINED'
    }
    
    for effect in effects:
        if effect.get('effect', '') in functional_effects:
            return True, effect.get('effect', '')
        # Also check by impact level
        if effect.get('impact', '') in ['HIGH', 'MODERATE']:
            return True, effect.get('effect', '')
    
    return False, ''

print("🔬 SNPeff effect parsing functions ready")

In [ ]:
def stream_snpeff_vcf_variants(vcf_path, target_chromosome, target_start, target_end):
    """Stream and process SNPeff-annotated VCF file for cyp51A variants"""
    
    variants = []
    total_variants = 0
    cyp51a_variants = 0
    functional_variants = 0
    synonymous_variants = 0
    
    print(f"📂 Processing: {vcf_path}")
    
    try:
        with open(vcf_path, 'r') as f:
            # Check for SNPeff in header
            has_snpeff = False
            for line in f:
                if line.startswith('#'):
                    if 'SnpEff' in line or 'ID=EFF' in line:
                        has_snpeff = True
                    continue
                
                # Process variant lines
                fields = line.strip().split('\t')
                if len(fields) < 8:
                    continue
                    
                chrom, pos, var_id, ref, alt, qual, filter_val, info = fields[:8]
                pos = int(pos)
                total_variants += 1
                
                # Filter by genomic region (cyp51A gene region with buffer)
                if chrom == target_chromosome and target_start - 1000 <= pos <= target_end + 1000:
                    # Parse SNPeff effects
                    effects = parse_snpeff_effects(info)
                    
                    # Check if this affects cyp51A
                    if is_cyp51a_variant(effects) or (target_start <= pos <= target_end):
                        cyp51a_variants += 1
                        
                        # Determine if functional
                        is_functional, effect_type = is_functional_variant(effects)
                        
                        if is_functional:
                            functional_variants += 1
                        else:
                            # Check for synonymous
                            is_synonymous = any(e.get('effect', '') == 'SYNONYMOUS_CODING' for e in effects)
                            if is_synonymous:
                                synonymous_variants += 1
                        
                        # Store variant info
                        variant = {
                            'chromosome': chrom,
                            'position': pos,
                            'ref': ref,
                            'alt': alt,
                            'quality': qual,
                            'effects': effects,
                            'is_functional': is_functional,
                            'effect_type': effect_type,
                            'local_coord': pos - target_start + 1 if target_start <= pos <= target_end else None
                        }
                        
                        variants.append(variant)
                
                # Progress indicator
                if total_variants % 50000 == 0:
                    print(f"  Processed {total_variants:,} variants, found {cyp51a_variants} in cyp51A region")
    
    except Exception as e:
        print(f"✗ Error processing {vcf_path}: {e}")
        return []
    
    # Summary statistics
    print(f"  ✓ Total variants: {total_variants:,}")
    print(f"  ✓ cyp51A region variants: {cyp51a_variants}")
    print(f"  ✓ Functional variants: {functional_variants}")
    print(f"  ✓ Synonymous variants: {synonymous_variants}")
    print(f"  ✓ SNPeff annotations: {'Yes' if has_snpeff else 'No'}")
    
    return variants

print("🔧 SNPeff VCF streaming function ready")

In [ ]:
# Process all SNPeff-annotated VCF files
all_sample_variants = {}
processing_summary = []

# Use the full list of SNPeff-annotated VCF HIDs
for hid in snpeff_vcf_ids:
    try:
        vcf_path = gxy.get(hid)
        if not vcf_path or '.vcf' not in vcf_path.lower():
            print(f"⚠️ HID {hid} is not a VCF file, skipping")
            continue
            
        # Extract sample name from filename
        sample_name = vcf_path.split('/')[-1].replace('.vcf', '').replace('_snpeff', '')
        
        print(f"\n🔬 Processing sample {sample_name} (HID {hid})")
        
        # Stream process the VCF file
        variants = stream_snpeff_vcf_variants(
            vcf_path,
            cyp51a_info['chromosome'],
            cyp51a_info['start'],
            cyp51a_info['end']
        )
        
        all_sample_variants[sample_name] = variants
        
        # Count functional vs synonymous
        functional_count = sum(1 for v in variants if v['is_functional'])
        synonymous_count = sum(1 for v in variants if any(e.get('effect', '') == 'SYNONYMOUS_CODING' for e in v['effects']))
        
        processing_summary.append({
            'sample': sample_name,
            'hid': hid,
            'total_variants': len(variants),
            'functional_variants': functional_count,
            'synonymous_variants': synonymous_count
        })
        
    except Exception as e:
        print(f"✗ Error processing HID {hid}: {e}")
        processing_summary.append({
            'sample': f'HID_{hid}',
            'hid': hid,
            'total_variants': 0,
            'functional_variants': 0,
            'synonymous_variants': 0,
            'error': str(e)
        })

print(f"\n📊 Processed {len(all_sample_variants)} samples successfully")

In [ ]:
# Display processing summary
print("\n📋 Processing Summary:")
print("Sample\t\tHID\tTotal\tFunctional\tSynonymous")
print("-" * 60)

for summary in processing_summary:
    sample = summary['sample'][:12].ljust(12)
    hid = str(summary['hid']).ljust(3)
    total = str(summary['total_variants']).ljust(5)
    functional = str(summary['functional_variants']).ljust(10)
    synonymous = str(summary['synonymous_variants']).ljust(10)
    
    print(f"{sample}\t{hid}\t{total}\t{functional}\t{synonymous}")
    
    if 'error' in summary:
        print(f"  ⚠️ Error: {summary['error']}")

# Check for suspicious patterns
functional_counts = [s['functional_variants'] for s in processing_summary if s['functional_variants'] > 0]
if functional_counts:
    unique_counts = set(functional_counts)
    if len(unique_counts) == 1 and list(unique_counts)[0] == 1000:
        print("\n⚠️ WARNING: All samples have exactly 1000 functional variants - this suggests artificial limits")
    else:
        print(f"\n✓ Variable functional variant counts: {min(functional_counts)}-{max(functional_counts)}")

## Step 4: Generate functional variant heatmap with proper SNPeff annotations

In [ ]:
# Create heatmap data matrix
if all_sample_variants:
    # Get all unique positions with functional variants
    all_functional_positions = set()
    
    for sample, variants in all_sample_variants.items():
        for variant in variants:
            if variant['is_functional'] and variant['local_coord']:
                all_functional_positions.add(variant['position'])
    
    if all_functional_positions:
        sorted_positions = sorted(all_functional_positions)
        samples = sorted(all_sample_variants.keys())
        
        print(f"📍 Creating heatmap for {len(sorted_positions)} functional positions across {len(samples)} samples")
        
        # Create matrix
        heatmap_data = np.zeros((len(samples), len(sorted_positions)))
        
        for i, sample in enumerate(samples):
            for variant in all_sample_variants[sample]:
                if variant['is_functional'] and variant['position'] in sorted_positions:
                    j = sorted_positions.index(variant['position'])
                    heatmap_data[i, j] = 1  # Mark presence of functional variant
        
        # Plot heatmap
        plt.figure(figsize=(max(12, len(sorted_positions) * 0.3), max(8, len(samples) * 0.3)))
        
        im = plt.imshow(heatmap_data, cmap='RdYlBu_r', aspect='auto')
        
        # Set labels
        plt.yticks(range(len(samples)), samples, fontsize=8)
        plt.xticks(range(len(sorted_positions)), 
                  [f"{pos}\n({pos - cyp51a_info['start'] + 1})" for pos in sorted_positions], 
                  rotation=45, ha='right', fontsize=6)
        
        plt.xlabel('Genomic Position (Local Coordinate)', fontsize=10)
        plt.ylabel('Samples', fontsize=10)
        plt.title('cyp51A Functional Variants Heatmap\n(SNPeff-annotated VCFs)', fontsize=12, pad=20)
        
        # Add colorbar
        plt.colorbar(im, label='Variant Present', shrink=0.8)
        
        plt.tight_layout()
        plt.show()
        
        print(f"✓ Heatmap generated with {np.sum(heatmap_data)} total functional variants")
        
    else:
        print("⚠️ No functional variants found in cyp51A region")
        
else:
    print("✗ No variant data available for heatmap generation")

In [ ]:
# Summary of key findings
print("\n🔍 Analysis Summary:")
print(f"Gene: cyp51A (Afu4g06890)")
print(f"Location: {cyp51a_info['chromosome']}:{cyp51a_info['start']}-{cyp51a_info['end']}")
print(f"Samples analyzed: {len(all_sample_variants)}")
print(f"VCF source: SNPeff-annotated collections #450 and #351")

if processing_summary:
    total_functional = sum(s['functional_variants'] for s in processing_summary)
    total_synonymous = sum(s['synonymous_variants'] for s in processing_summary)
    print(f"Total functional variants: {total_functional}")
    print(f"Total synonymous variants: {total_synonymous}")
    
    # Show top variant positions
    if all_functional_positions:
        print(f"\nFunctional variant positions:")
        for pos in sorted(list(all_functional_positions)[:10]):
            local_coord = pos - cyp51a_info['start'] + 1
            print(f"  {pos} (local: {local_coord})")
            
print("\n✅ Analysis complete using SNPeff-annotated VCF files")